# 📊 Dokumentasi: Scraping Ulasan Shopee + Text Processing

## 🎯 Gambaran Umum
Project ini terdiri dari 2 tahap utama:
1. **Scraping Ulasan** - Mengambil ulasan produk dari Shopee secara otomatis
2. **Text Processing** - Pembersihan dan preprocessing teks ulasan untuk analisis NLP

---

## 📋 Prerequsite & Setup

### Instalasi Library
```bash
pip install undetected-chromedriver pandas selenium webdriver-manager sastrawi openpyxl
```

### File yang Dibutuhkan
- Tidak ada file input untuk scraping (buat dari nol)
- Untuk tahap 2, butuh file Excel hasil scraping

---

## 🔄 Tahap 1: Scraping Ulasan Shopee

### Fungsi: `scrape_shopee_tembak_dalam(url_produk, total_halaman=3)`

**Input:**
- `url_produk` (str): Link produk Shopee
- `total_halaman` (int): Jumlah halaman ulasan yang diambil (default: 3)

**Output:**
- File CSV: `dataset_ulasan.csv`
- Kolom: `Username`, `Rating`, `Komentar`

**Cara Kerja:**
1. Buka browser menggunakan undetected ChromeDriver (untuk bypass deteksi bot)
2. User perlu scroll manual ke area ulasan terlebih dahulu
3. Ambil data ulasan dengan class selectors:
   - `.InK5kS` → nama username
   - `.YNedDV` → isi komentar dan rating
4. Loop untuk halaman berikutnya menggunakan tombol Next
5. Simpan hasil ke CSV

**Catatan:**
- Memerlukan interaksi manual (scroll awal dan confirm dengan 'y')
- Random delay 5-8 detik antar halaman untuk terhindar throttling
- Jika ada error koneksi, coba gunakan hotspot HP

---

## 🧹 Tahap 2: Text Processing & Pembersihan Data

### Fungsi: `bersihkan_teks(teks)`

**Input:**
- `teks` (str): Text ulasan mentah

**Output:**
- Teks yang sudah dibersihkan dan distandarisasi

**Tahapan Pembersihan:**

| Step | Proses | Contoh |
|------|--------|---------|
| 1. Case Folding | Ubah ke huruf kecil | "MANTAP!" → "mantap!" |
| 2. Cleaning | Hapus angka, emoji, link | "Bagus!! 😍👍" → "bagus" |
| 3. Stopword Removal | Hapus kata dasar Bahasa Indonesia | "yang di sini" → "sini" |
| 4. Stemming | Ubah ke bentuk dasar | "pengiriman" → "kirim" |

**Contoh Transformasi:**
```
Input:  "BARANGNYA MANTAP!! Pengiriman cepat banget 😍👍 Buruan beli!"
Output: "barang mantap pengiriman cepat buruan beli"
```

**Output:**
- File Excel: `Dataset_NLP_Siap_Olah.xlsx`
- Kolom tambahan: `Komentar_Bersih`
- Baris kosong otomatis dihapus

---

## 📊 Flow Diagram

```
┌─────────────────┐
│  Shopee URL     │
└────────┬────────┘
         │
         ▼
┌─────────────────────────────┐
│  Scrape Ulasan             │
│  (undetected_chromedriver)  │
└────────┬────────────────────┘
         │
         ▼
┌──────────────────────┐
│ dataset_ulasan.csv   │ ◄─── Raw Data (Username, Rating, Komentar)
└────────┬─────────────┘
         │
         ▼
┌──────────────────────────────────┐
│  Text Processing                 │
│  (Sastrawi stemmer + stopword)    │
└────────┬─────────────────────────┘
         │
         ▼
┌──────────────────────────────────┐
│ Dataset_NLP_Siap_Olah.xlsx       │ ◄─── Clean Data (untuk NLP)
└──────────────────────────────────┘
```

---

## ⚙️ Konfigurasi & Parameter

### Scraping
- `total_halaman`: Ubah jumlah halaman yang diambil (default 3)
- Delay antar halaman: 5-8 detik (random)
- Selectors bisa berubah jika Shopee update struktur HTML

### Text Processing
- Menggunakan Sastrawi default (Indonesian language)
- Otomatis hapus baris kosong setelah cleaning
- Encoding: UTF-8 (mendapat emoji & karakter khusus)

---

## 🐛 Troubleshooting

| Error | Solusi |
|-------|--------|
| "Browser closed" | Pastikan Chrome terinstall dan webdriver kompatibel |
| 0 ulasan ter-scrape | Scroll manual lebih bawah sebelum confirm, atau Shopee membatasi IP |
| Class selector berubah | Inspect element di Shopee, update `.InK5kS` dan `.YNedDV` |
| Memory error | Kurangi `total_halaman` atau restart kernel |
| File "xlsx" tidak bisa dibuka | Pastikan `openpyxl` sudah terinstall |

---

## 📝 Contoh Penggunaan

```python
# 1. SCRAPING
URL = "https://shopee.co.id/Ms-Glow-for-Men-..."
scrape_shopee_tembak_dalam(URL, total_halaman=5)

# 2. TEXT PROCESSING
# (Otomatis baca dataset_ulasan.csv → bersihkan → simpan ke xlsx)
```

---

## 📁 Output Files

1. **dataset_ulasan.csv**
   - Raw data dari scraping
   - 3 kolom: Username, Rating, Komentar

2. **Dataset_NLP_Siap_Olah.xlsx**
   - Data yang sudah dibersihkan
   - 4 kolom: Username, Rating, Komentar, Komentar_Bersih
   - Siap untuk analisis sentimen atau topik modeling

In [ ]:
import undetected_chromedriver as uc
import pandas as pd
import time
import random
from selenium.webdriver.common.by import By

def scrape_shopee_tembak_dalam(url_produk, total_halaman=3):
    print("Membuka browser... Gass poll!")
    driver = uc.Chrome()
    
    try:
        driver.get(url_produk)
        
        print("\n" + "="*40)
        print("LAKUKAN INI:")
        print("1. Scroll manual sampai ulasan ke-5 atau lebih muncul di layar.")
        print("2. Balik sini, ketik 'y' dan Enter.")
        print("="*40)
        
        if input("Siap? (y/n): ").lower() != 'y': return

        all_data = []
        for p in range(total_halaman):
            print(f"\n--- Proses Halaman {p+1} ---")
            
            # Scroll halus biar gambar dan teks ke-load
            for _ in range(5):
                driver.execute_script("window.scrollBy(0, 400);")
                time.sleep(1)
            
            time.sleep(2) 

            # Langsung ke isi ulasan bukan ke wrapper wong sok error
            users = driver.find_elements(By.CLASS_NAME, "InK5kS")
            contents = driver.find_elements(By.CLASS_NAME, "YNedDV")
            
            print(f"Mantap! Ketemu {len(users)} nama user dan {len(contents)} isi ulasan.")

            # gabung data sesuai indeks
            for i in range(len(users)):
                try:
                    nama = users[i].text
                    
                    # Cari jumlah bintang di dalam blok konten tersebut
                    bintang = len(contents[i].find_elements(By.CLASS_NAME, "icon-rating-solid"))
                    
                    # Ambil semua teks komentar
                    komen = contents[i].text.replace('\n', ' ')
                    
                    # Hanya simpan kalau ada teks komentarnya
                    if komen.strip():
                        all_data.append({
                            'Username': nama,
                            'Rating': bintang if bintang > 0 else 5, 
                            'Komentar': komen
                        })
                except:
                    continue
            
            try:
                next_btn = driver.find_element(By.CSS_SELECTOR, "button.shopee-icon-button--right")
                driver.execute_script("arguments[0].click();", next_btn)
                print("Pindah ke halaman berikutnya...")
                time.sleep(random.uniform(5, 8))
            except:
                print("Tombol Next tidak ditemukan atau sudah habis.")
                break

        if all_data:
            df = pd.DataFrame(all_data)
            df.to_csv('dataset_ulasan.csv', index=False)
            print(f"\nBOOM! {len(all_data)} ulasan berhasil diamankan ke dataset_ulasan.csv")
        else:
            print("\nMasih 0 juga? Waduh, coba ganti koneksi internet (pakai hotspot HP), kadang IP dibatasi Shopee.")

    finally:
        driver.quit()

# GAS!
URL_TARGET = "{URL_PRODUK_SHOPEE}"
scrape_shopee_tembak_dalam(URL_TARGET, total_halaman=3)

Membuka browser... Gass poll, Gung!

LAKUKAN INI:
1. Scroll manual sampai ulasan ke-5 atau lebih muncul di layar.
2. Balik sini, ketik 'y' dan Enter.

--- Proses Halaman 1 ---
Mantap! Ketemu 6 nama user dan 6 isi ulasan.
Pindah ke halaman berikutnya...

--- Proses Halaman 2 ---
Mantap! Ketemu 6 nama user dan 6 isi ulasan.
Pindah ke halaman berikutnya...

--- Proses Halaman 3 ---
Mantap! Ketemu 6 nama user dan 6 isi ulasan.
Pindah ke halaman berikutnya...

BOOM! 15 ulasan berhasil diamankan ke dataset_ulasan_agung.csv


In [ ]:
import pandas as pd
import re
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

print("Mulai proses pembersihan data... Tunggu bentar ya!")

# 1. Load Data Excel
file_excel = 'Data_Ulasan_Shopee_Rapi.xlsx'
df = pd.read_excel(file_excel)

# Siapkan alat pembersih dari Sastrawi
stopword_factory = StopWordRemoverFactory()
stopword_remover = stopword_factory.create_stop_word_remover()

stemmer_factory = StemmerFactory()
stemmer = stemmer_factory.create_stemmer()

def bersihkan_teks(teks):
    # Cek kalau teksnya kosong (NaN)
    if pd.isna(teks):
        return ""
        
    teks = str(teks)
    
    # A. Case Folding (Ubah ke huruf kecil semua)
    teks = teks.lower()
    
    # B. Cleaning (Hapus angka, tanda baca, emoji, link)
    teks = re.sub(r'http\S+', '', teks) # Hapus link
    teks = re.sub(r'[^a-z\s]', '', teks) # Hanya sisakan huruf a-z dan spasi
    teks = re.sub(r'\s+', ' ', teks).strip() # Hapus spasi berlebih
    
    # C. Stopword Removal (Hapus kata hubung kayak "yang", "di", "dan")
    teks = stopword_remover.remove(teks)
    
    # D. Stemming (Ubah kata berimbuhan jadi kata dasar, misal "pengiriman" -> "kirim")
    teks = stemmer.stem(teks)
    
    return teks

df['Komentar_Bersih'] = df['Komentar'].apply(bersihkan_teks)

df = df[df['Komentar_Bersih'] != '']

# Simpan ke Excel Baru
file_hasil = 'Dataset_NLP_Siap_Olah.xlsx'
df.to_excel(file_hasil, index=False, engine='openpyxl')

print("\nMantap! Data udah kinclong.")
print("Cek 3 data teratas:")
print(df[['Komentar', 'Komentar_Bersih']].head(3))

Mulai proses pembersihan data... Tunggu bentar ya, Gung!

Mantap! Data udah kinclong.
Cek 3 data teratas:
                                            Komentar  \
0  Alhamdulillah pengiriman nya cepet..  Saya sdh...   
1  Barang telah sampai dengan selamat Toko Amanah...   
2  Pengiriman cepat barang sesuai dengan pesanan ...   

                                     Komentar_Bersih  
0  alhamdulillah kirim nya cepet sdh kali sen alh...  
1  barang sampai selamat toko amanah langgan teru...  
2       kirim cepat barang sesuai pesan terima kasih  
